# Paper LLM scorer

Structured LLM scorer imported from `urbanomy.methods.agent`.


In [ ]:
from urbanomy.methods.agent import SingleAgentBaseline, init_llm
from urbanomy.methods.land_value_modeling import Evaluation

llm = init_llm("deepseek/deepseek-v4-flash")
baseline = SingleAgentBaseline(llm=llm, output_schema=Evaluation)

In [ ]:
import geopandas as gpd
basline_blocks = gpd.read_file('./data/blocks_agg_with_indicators.geojson')
basline_blocks.head()

In [ ]:
basline_blocks.iloc[0]

In [ ]:
from catboost import CatBoostRegressor
model = CatBoostRegressor()
model.load_model('./data/catboost_land_value_no_services.cbm')  # модель на лог-цене
print(len(model.feature_names_))

In [ ]:
feature_cols = [
'residential','business','recreation','industrial','transport','special',
'agriculture','land_use','share','footprint_area','build_floor_area',
'living_area','non_living_area','population','site_area','fsi','gsi',
'mxi','l','morphotype','area_accessibility'
]
cat_features = ['land_use', 'morphotype']
numeric_feats = [c for c in feature_cols if c not in cat_features]
basline_blocks["id"] = basline_blocks.index

In [ ]:
basline_blocks['residential'] = basline_blocks['residential'].astype('float64')

In [ ]:
# import geopandas as gpd
# import pandas as pd

# # 1. Переводим в метрическую CRS
# baseline_blocks_metric = basline_blocks.to_crs(basline_blocks.estimate_utm_crs()).copy()

# # 2. Считаем площадь в м2 и гектарах
# baseline_blocks_metric["calc_area_m2"] = baseline_blocks_metric.geometry.area
# baseline_blocks_metric["calc_area_ha"] = baseline_blocks_metric["calc_area_m2"] / 10_000

# # 3. Переводим site_area в гектары
# baseline_blocks_metric["site_area_ha"] = baseline_blocks_metric["site_area"] / 10_000

# # 4. Сравниваем
# baseline_blocks_metric["area_diff_ha"] = (
#     baseline_blocks_metric["calc_area_ha"] - baseline_blocks_metric["site_area_ha"]
# )

# baseline_blocks_metric["area_diff_pct"] = (
#     baseline_blocks_metric["area_diff_ha"] / baseline_blocks_metric["site_area_ha"] * 100
# )

# # 5. Сводка
# print(
#     baseline_blocks_metric[
#         ["site_area_ha", "calc_area_ha", "area_diff_ha", "area_diff_pct"]
#     ].describe()
# )

# # 6. Самые большие расхождения
# comparison_ha = baseline_blocks_metric[
#     ["site_area_ha", "calc_area_ha", "area_diff_ha", "area_diff_pct", "geometry"]
# ].copy()

# comparison_ha["abs_area_diff_pct"] = comparison_ha["area_diff_pct"].abs()
# comparison_ha.sort_values("abs_area_diff_pct", ascending=False).head(10)


In [ ]:
# удаляем старую колонку
basline_blocks = basline_blocks.drop(columns=["site_area"], errors="ignore").copy()

# пересчитываем площадь в метрической CRS
basline_blocks_metric = basline_blocks.to_crs(basline_blocks.estimate_utm_crs()).copy()
basline_blocks["site_area"] = basline_blocks_metric.geometry.area.values


In [ ]:
from urbanomy.methods.land_value_modeling import LandPriceEstimator

estimator = LandPriceEstimator(
    model=model,
    orig_features=numeric_feats+cat_features,
    categorical_features=cat_features,
    blocks=basline_blocks, #or blocks_198
)
blocks_pred = estimator.predict()
blocks_pred.head()


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
# 1) Цена за сотку (100 м²)
blocks_pred["land_value_per_100m2"] = blocks_pred["land_value"] / blocks_pred["site_area"] * 100

# 2) Заменяем inf на NaN
blocks_pred = blocks_pred.replace([np.inf, -np.inf], np.nan)
blocks_pred = blocks_pred.fillna(0)

# 3) Удаляем выбросы: оставляем данные до 99-го перцентиля
p99 = blocks_pred["land_value_per_100m2"].quantile(0.95)
blocks_clean = blocks_pred[blocks_pred["land_value_per_100m2"] <= p99].copy()

print(blocks_clean["land_value_per_100m2"].describe())

# 4) Гистограмма после очистки
plt.figure()
blocks_clean["land_value_per_100m2"].dropna().hist(bins=50)
plt.xlabel("Цена за сотку (руб.)")
plt.ylabel("Частота")
plt.title("Распределение цены за сотку")
plt.show()



In [ ]:
import matplotlib.pyplot as plt

blocks_clean.plot(
    column='land_value_per_100m2',
    legend=True,
    figsize=(20,20),
    cmap='coolwarm',
    edgecolor='black',   # <-- цвет границы
    linewidth=0.2        # <-- толщина границы
).set_axis_off()
plt.title('Карта стоимости земельных участков за сотку (руб.)', fontsize=16)

plt.show()

In [ ]:
import matplotlib.pyplot as plt

blocks_clean.plot(
    column='land_value',
    legend=True,
    figsize=(20,20),
    cmap='coolwarm',
    edgecolor='black',   # <-- цвет границы
    linewidth=0.2        # <-- толщина границы
).set_axis_off()
plt.title('Карта стоимости земельных участков (руб.)', fontsize=16)

plt.show()

# Выбор сценария развития

In [ ]:
blocks_clean["id"] = blocks_clean.index

In [ ]:
target_id = 86

In [ ]:
import matplotlib.pyplot as plt

target_id = 86
target_block = blocks_clean.loc[blocks_clean["id"] == target_id]

fig, ax = plt.subplots(figsize=(25, 35))
blocks_clean.plot(ax=ax, color="lightgrey", edgecolor="white", linewidth=0.6)
target_block.plot(ax=ax, color="none", edgecolor="gold", linewidth=5.5)
target_block.centroid.plot(ax=ax, color="red", markersize=30, zorder=3)

ax.set_title(f"Изменяемый квартал (id={target_id})")
ax.axis("off")
plt.show()


In [ ]:
# blocks_after.to_file('./data/test/gatchina_RESIDENTIAL.geojson', driver='GeoJSON')

# Оптимизация района

## NSGA 2

In [ ]:
prompts = {
    1: """
Evaluate the presented scenario based on the criterion of social inclusiveness and low displacement risk.
Determine whether the scenario preserves the social and functional diversity of the area, access to everyday services, and opportunities for vulnerable user groups.
Assign a score from 0 to 1.
""".strip(),
    2: """
Evaluate the presented scenario based on the criterion of strategic relevance for the long-term development of the city and district.
Consider compactness, functional mix, the balance of housing, jobs, and services, and the area's ability to adapt to future changes.
Assign a score from 0 to 1.
""".strip(),
    3: """
Evaluate the presented scenario based on its ability to create conditions for active urban life.
Consider functional mix, the potential for active ground floors, pedestrian accessibility, public spaces, and diverse everyday use scenarios throughout the day and evening.
Assign a score from 0 to 1.
""".strip(),
    4: """
Evaluate the presented scenario based on the criterion of aligning the interests of various land users and minimizing conflict risk.
Analyze potential conflicts between residents, pedestrians, drivers, business owners, visitors, employees, seniors, and families with children.
Assign a score from 0 to 1.
""".strip(),
    5: """
Evaluate the presented scenario based on the criterion of environmental and infrastructure sustainability.
Consider building density, transport and infrastructure load, resource efficiency, environmental consequences, and the possibility of phased adaptation of the area.
Assign a score from 0 to 1.
""".strip(),
}

In [ ]:
from pathlib import Path

from urbanomy.methods.land_value_modeling import (
    build_pareto_front_dataframe,
    run_nsga2_with_strategic_alignment,
)

site_area = float(blocks_clean.loc[blocks_clean["id"] == target_id, "site_area"].iloc[0])

constraints = {
    "footprint_area": {"type": "float", "min": 0.0, "max": 0.1 * site_area},
    "l": {"type": "float", "min": 1.0, "max": 10.0},
    "mxi": {"type": "float", "min": 0.1, "max": 1.0},
    "residential": {"type": "float", "min": 0.0, "max": 1.0},
    "business": {"type": "float", "min": 0.0, "max": 1.0},
    "recreation": {"type": "float", "min": 0.0, "max": 1.0},
    "industrial": {"type": "float", "min": 0.0, "max": 1.0},
    "transport": {"type": "float", "min": 0.0, "max": 1.0},
    "special": {"type": "float", "min": 0.0, "max": 1.0},
    "agriculture": {"type": "float", "min": 0.0, "max": 1.0},
}

notebook_dir = Path("examples") if Path("examples/NSGA2_LLM.ipynb").exists() else Path(".")
results_dir = notebook_dir / "nsga_2_llm_prompts"

seeds = [38, 39, 40, 41]
json_rule = '\nReturn only valid JSON, no markdown: {"score": 0.0}'

for seed in seeds:
    seed_dir = results_dir / f"seed_{seed}"
    seed_dir.mkdir(parents=True, exist_ok=True)

    for prompt_id, prompt in prompts.items():
        prompt_dir = seed_dir / f"prompt_{prompt_id}"
        prompt_dir.mkdir(parents=True, exist_ok=True)

        optimization_log_path = prompt_dir / f"prompt_{prompt_id}_optimization_log.jsonl"
        pareto_front_path = prompt_dir / f"prompt_{prompt_id}_pareto_front.jsonl"

        if optimization_log_path.exists() and pareto_front_path.exists():
            print(f"skip seed={seed}, prompt={prompt_id}")
            continue

        print(f"run seed={seed}, prompt={prompt_id}")

        res, problem = run_nsga2_with_strategic_alignment(
            blocks=blocks_clean,
            model=model,
            estimator_kwargs={
                "orig_features": numeric_feats + cat_features,
                "categorical_features": cat_features,
            },
            constraints=constraints,
            target_id=target_id,
            baseline=baseline,
            prompt=prompt + json_rule,
            pop_size=20,
            n_gen=30,
            seed=seed,
            verbose=True,
            save_history=True,
            log_optimization=True,
            optimization_log_path=optimization_log_path,
        )

        pareto_prompt_df = build_pareto_front_dataframe(res=res, problem=problem)
        pareto_prompt_df.insert(0, "seed", seed)
        pareto_prompt_df.insert(1, "prompt_id", prompt_id)
        pareto_prompt_df.to_json(
            pareto_front_path,
            orient="records",
            lines=True,
            force_ascii=False,
        )

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

optimizer = problem

# 1) Базовая стоимость "до"
baseline_land_value = optimizer.evaluate_catboost(
    geonome=optimizer.blocks,
    model=optimizer.model,
    orig_features=optimizer.estimator_kwargs["orig_features"],
    cat_features=optimizer.estimator_kwargs["categorical_features"],
    radius_list=None,
)

# 2) Собираем ВСЕ решения: из history (если optimize(..., save_history=True)), иначе из res.X/res.F
X_all, F_all = [], []
if getattr(res, "history", None):
    for alg in res.history:
        pop = alg.pop
        Xi, Fi = pop.get("X"), pop.get("F")
        if Xi is not None and Fi is not None and len(Xi):
            X_all.append(np.asarray(Xi))
            F_all.append(np.asarray(Fi))

if X_all:
    X = np.vstack(X_all)
    F = np.vstack(F_all)
else:
    X = np.asarray(res.X)
    F = np.asarray(res.F)

# 3) Цели
land_value_total = -F[:, 0]
admin_gain = land_value_total - baseline_land_value
investor_npv = -F[:, 1]

# 4) LANDUSE для каждого решения (через тот же repair, что в оптимизаторе)
landuse_labels = []
for genome_vec in X:
    changes = {name: genome_vec[j] for j, name in enumerate(optimizer.var_names)}
    repaired = optimizer._repair_genome(changes)   # использует логику из вашей задачи
    lu = repaired["land_use"]
    lu_name = getattr(lu, "name", str(lu).split(".")[-1])
    landuse_labels.append(lu_name)

landuse_labels = np.array(landuse_labels)
unique_lu = np.unique(landuse_labels)

# 5) Цвета + легенда
cmap = plt.get_cmap("tab10", len(unique_lu))
color_map = {lu: cmap(i) for i, lu in enumerate(unique_lu)}

plt.figure(figsize=(10, 6))

for lu in unique_lu:
    m = landuse_labels == lu
    plt.scatter(
        investor_npv[m],
        admin_gain[m],
        s=35,
        alpha=0.8,
        color=color_map[lu],
        label=f"LANDUSE: {lu}",
    )

# (опционально) поверх — линия самого Pareto-фронта из res.F
pf_x = -res.F[:, 1]
pf_y = -res.F[:, 0] - baseline_land_value
order = np.argsort(pf_x)
plt.plot(pf_x[order], pf_y[order], color="black", lw=1.5, alpha=0.7, label="Pareto front")

plt.xlabel("NPV инвестора (X), руб.")
plt.ylabel("Прирост общей стоимости земли (Y), руб.")
plt.title("Парето фронт")
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()   

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

SEED = 38  # укажите нужный seed

notebook_dir = Path("examples") if Path("examples/paper.ipynb").exists() else Path(".")
seed_dir = notebook_dir / "nsga_2_llm_prompts" / f"seed_{SEED}"
no_llm_path = notebook_dir / "nsga_2_without_llm" / f"no_llm_pareto_front_{SEED}.jsonl"

front_files = sorted(
    seed_dir.glob("prompt_*/prompt_*_pareto_front.jsonl"),
    key=lambda path: int(path.parent.name.rsplit("_", 1)[-1]),
)

if not front_files:
    raise FileNotFoundError(f"Нет LLM pareto-фронтов для seed {SEED} в {seed_dir}")
if not no_llm_path.exists():
    raise FileNotFoundError(no_llm_path)

plt.figure(figsize=(11, 7))
cmap = plt.get_cmap("tab10", len(front_files))

for i, path in enumerate(front_files):
    prompt_id = int(path.parent.name.rsplit("_", 1)[-1])
    df = pd.read_json(path, lines=True).sort_values("investor_npv")

    plt.plot(
        df["investor_npv"],
        df["land_value_gain"],
        marker="D",
        ms=4,
        lw=1.8,
        alpha=0.85,
        color=cmap(i),
        label=f"LLM prompt {prompt_id}",
    )

no_llm_df = pd.read_json(no_llm_path, lines=True).sort_values("investor_npv")
plt.plot(
    no_llm_df["investor_npv"],
    no_llm_df["land_value_gain"],
    marker="o",
    ms=5,
    lw=2.5,
    color="black",
    label=f"no LLM seed {SEED}",
)

plt.xlabel("NPV инвестора (X), руб.")
plt.ylabel("Прирост общей стоимости земли (Y), руб.")
plt.title(f"Pareto fronts: LLM prompts и no-LLM для seed {SEED}")
plt.grid(True, alpha=0.3)
plt.legend(title="Запуск")
plt.tight_layout()
plt.show()